In [1]:
from openai import OpenAI
import os
from tinydb import TinyDB

In [2]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key= os.getenv("OPENROUTER_API_KEY")
) 
model = "openai/gpt-oss-120b"

In [ ]:
from hypotheses.HypothesisRefiner import HypothesisRefiner

hypothesesDb = TinyDB("../store/frozenlake/hypotheses.json")
 
envCurator = HypothesisRefiner(client=client, model=model, hypothesesDb=hypothesesDb)

debugTrajectory = '''
== Step 1 ==
== Response: 
== Num_Tools: 1
== Tool: move_down
== Tool Parameters: {}
== New State:
  S  F  F  F 
[F] F  F  H 
 H  F  F  H 
 F  H  F  F 
 F  F  F  G 
======
== Step 2 ==
== Response: 
== Step 3 ==
== Response: 
== Num_Tools: 1
== Tool: move_right
== Tool Parameters: {}
== New State:
 [S] F  F  F 
 F  F  F  H 
 H  F  F  H 
 F  H  F  F 
 F  F  F  G 
======
== Step 4 ==
== Response: Proceeding towards the nearest H by moving down.
== Num_Tools: 1
== Tool: move_down
== Tool Parameters: {}
== New State:
  S  F  F  F 
[F] F  F  H 
 H  F  F  H 
 F  H  F  F 
 F  F  F  G 
======
== Step 5 ==
== Response: Moving right towards the goal while avoiding H.
== Num_Tools: 1
== Tool: move_right
== Tool Parameters: {}
== New State:
  S  F  F  F 
 F  F  F  H 
[H] F  F  H 
 F  H  F  F 
 F  F  F  G 
======
'''

envCurator.run(debugTrajectory, debug=True)

In [3]:
from environments.EnvironmentFrozenLake import EnvironmentFrozenLake

hypothesesDb = TinyDB("../store/frozenlake/hypotheses.json")
policyDb = TinyDB("../store/frozenlake/policies.json")

env = EnvironmentFrozenLake(policyDb=policyDb, hyptothesisDb=hypothesesDb)
env.reset()

print(env.getInitialGeneratorPrompt())
print(env.getGeneratorTools())

env.moveLeft()
print(env.getState())

env.moveDown()
print(env.getState())

env.moveRight()
print(env.getState())


print(env.getInitialGeneratorPrompt())

[{'role': 'system', 'content': "\n                     You are a multi-turn LLM Agent Navigator in a dynamic 2D environment. Your goal: reach position G from current position [] using the provided tools for movement.\n\n                     ## Multi-Turn Operation\n                     You receive a NAVIGATION TRAJECTORY containing:\n                     - All your previous decisions and reasoning\n                     - Environment responses after each action\n                     - Current state resulting from your last move\n\n                     Your output becomes input for your next iteration. Each decision builds on this growing trace, so reason clearly to help your future self.\n\n                     ## Core Task\n                     Read the playbook and the reflection -> Apply rules, knowledge and strategies retrieved from those documents -> Decide the next one move from the given context\n\n                     ## Decision Process\n                     1. **Apply Learning